In [1]:
!pip install face_recognition

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566164 sha256=372d5ff977920d34f5f0e4b3f2539fe9ba835ef3360e9e589578e6aaf3c746f6
  Stored in directory: /root/.cache/pip/wheels/7a/eb/cf/e9eced74122b679557f597bb7c8e4c739cfcac526db1fd523d
Successfully built face-recognition-models


In [ ]:
import os
import cv2
import face_recognition
import json

In [ ]:
diretorio_imagens = "/content/drive/MyDrive/Civil Patrick e Isabela - @stalonemoreira"

In [ ]:
imagens = [imagem for imagem in os.listdir(diretorio_imagens) if imagem.lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))]

In [ ]:
len(imagens)

262

In [ ]:
# Função para carregar codificações faciais conhecidas
def carregar_codificacoes_conhecidas(imagens, diretorio_imagens):
    rostos_conhecidos = []
    pessoas_correspondentes = []

    for imagem_nome in imagens:
        caminho_imagem = os.path.join(diretorio_imagens, imagem_nome)
        imagem = face_recognition.load_image_file(caminho_imagem)
        rostos_na_imagem = face_recognition.face_locations(imagem)

        for rosto_na_imagem in rostos_na_imagem:
            rosto_conhecido = face_recognition.face_encodings(imagem, [rosto_na_imagem])[0]
            rostos_conhecidos.append(rosto_conhecido)
            pessoas_correspondentes.append(len(pessoas_correspondentes) + 1)

    return rostos_conhecidos, pessoas_correspondentes

In [ ]:
# Função para identificar e desenhar retângulos em rostos nas imagens
def identificar_desenhar_rostos(imagens, diretorio_imagens, rostos_conhecidos, pessoas_correspondentes):
    dados_pessoas = {}

    for pessoa_id, imagem_nome in enumerate(imagens, start=1):
        caminho_imagem = os.path.join(diretorio_imagens, imagem_nome)
        imagem = cv2.imread(caminho_imagem)

        # Encontrando rostos na imagem
        rostos_na_imagem = face_recognition.face_locations(imagem)
        rostos_na_imagem_encodings = face_recognition.face_encodings(imagem, rostos_na_imagem)

        # Conjunto para controlar as pessoas já identificadas na imagem
        pessoas_identificadas = set()

        # Iterando sobre os rostos na imagem
        for i, rosto_na_imagem_encoding in enumerate(rostos_na_imagem_encodings):
            correspondencias = face_recognition.compare_faces(rostos_conhecidos, rosto_na_imagem_encoding)
            pessoa_correspondente = pessoas_correspondentes[correspondencias.index(True)]

            # Adicionando informações ao dicionário apenas se a pessoa não foi identificada anteriormente
            if pessoa_correspondente not in pessoas_identificadas:
                chave_pessoa = f'Pessoa {pessoa_correspondente}'
                if chave_pessoa not in dados_pessoas:
                    dados_pessoas[chave_pessoa] = {'apareceu nas fotos': set()}
                dados_pessoas[chave_pessoa]['apareceu nas fotos'].add(imagem_nome)

                # Adicionando a pessoa ao conjunto de pessoas identificadas
                pessoas_identificadas.add(pessoa_correspondente)

    # Convertendo conjuntos de fotos de volta para listas no JSON
    for chave_pessoa, dados in dados_pessoas.items():
        dados['apareceu nas fotos'] = list(dados['apareceu nas fotos'])

    # Salvando dados em um arquivo JSON
    salvar_dados_em_json(dados_pessoas)

In [ ]:
# Função para salvar os dados em um arquivo JSON
def salvar_dados_em_json(dados):
    with open('agrupamento_por_rosto.json', 'w') as json_file:
        json.dump(dados, json_file, indent=4)

In [ ]:
rostos_conhecidos, pessoas_correspondentes = carregar_codificacoes_conhecidas(imagens, diretorio_imagens)
identificar_desenhar_rostos(imagens, diretorio_imagens, rostos_conhecidos, pessoas_correspondentes)